# GPT-SoVITS 全自动训练 - 达妮娅语音

数据集: https://huggingface.co/datasets/huanx/daniya-voice-gptsovits

按顺序运行所有单元格。**运行前务必确认已开启 GPU (T4)**。

## Step 0: 环境检测

In [ ]:
import os, sys

# 检测环境
if os.path.exists('/kaggle'):
    WORK_DIR = '/kaggle/working'
    print('Kaggle 环境')
else:
    WORK_DIR = '/content'
    print('Colab 环境')

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'工作目录: {WORK_DIR}')

# 检查 GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('ERROR: 没有检测到 GPU！请在 运行时->更改运行时类型 中选择 GPU T4')
    sys.exit(1)

## Step 1: 克隆仓库并安装依赖

In [ ]:
REPO_DIR = f'{WORK_DIR}/GPT-SoVITS'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'仓库目录: {REPO_DIR}')

# 安装依赖 (opencc 可能编译失败，忽略)
!pip install -r requirements.txt 2>&1 | tail -3
!pip install huggingface_hub 2>&1 | tail -3

print('Step 1 完成')

## Step 2: 下载预训练模型

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

os.chdir(REPO_DIR)

if os.path.exists('pretrained_models/chinese-roberta-wwm-ext-large'):
    print('预训练模型已存在，跳过下载')
else:
    print('下载 pretrained_models.zip ...')
    path = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                            filename='pretrained_models.zip',
                            local_dir='.')
    print('解压中 ...')
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall('.')

# 检查
print(f'pretrained_models/ 内容:')
for f in sorted(os.listdir('pretrained_models')):
    print(f'  {f}')

# 下载 G2PW
if not os.path.exists('GPT_SoVITS/text/g2pw'):
    print('\n下载 G2PWModel.zip ...')
    g2pw = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                           filename='G2PWModel.zip', local_dir='.')
    with zipfile.ZipFile(g2pw, 'r') as z:
        z.extractall('GPT_SoVITS/text/')

print('Step 2 完成')

## Step 3: 下载数据集

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = f'{WORK_DIR}/daniya_dataset'

print('下载达妮娅数据集 ...')
snapshot_download(repo_id='huanx/daniya-voice-gptsovits',
                   repo_type='dataset',
                   local_dir=DATASET_DIR)

wav_count = len(list(Path(f'{DATASET_DIR}/audio').glob('*.wav')))
print(f'Step 3 完成: {wav_count} 个音频')

## Step 4: 准备数据目录

In [ ]:
import pandas as pd

os.chdir(REPO_DIR)

EXP_NAME = 'daniya'
EXP_DIR = os.path.abspath(f'output/{EXP_NAME}')

# 创建目录结构
for sub in ['5-wav32k', '1_16k_wavs', '1_32k_wavs', '2a-asr',
            '2b-feature', '3-bert', '4-cnhubert', '6-name2semantic',
            'logs_s1', 'logs_s2_v2', 'weights']:
    os.makedirs(f'{EXP_DIR}/{sub}', exist_ok=True)

# 复制音频
wav_files = sorted(Path(f'{DATASET_DIR}/audio').glob('*.wav'))
for f in wav_files:
    shutil.copy2(f, f'{EXP_DIR}/5-wav32k/')
    shutil.copy2(f, f'{EXP_DIR}/1_16k_wavs/')
    shutil.copy2(f, f'{EXP_DIR}/1_32k_wavs/')
print(f'复制 {len(wav_files)} 个音频')

# 生成 .list 文件
metadata = pd.read_csv(f'{DATASET_DIR}/metadata.csv')
list_lines = []
for _, row in metadata.iterrows():
    wav_path = os.path.abspath(f'{EXP_DIR}/5-wav32k/{row["file"]}')
    text = str(row['text']).strip()
    list_lines.append(f'{wav_path}|{EXP_NAME}|zh|{text}')

INP_TEXT = os.path.abspath(f'{EXP_DIR}/2a-asr/{EXP_NAME}.list')
with open(INP_TEXT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(list_lines) + '\n')

print(f'生成 {len(list_lines)} 条训练记录')
print(f'示例: {list_lines[0][:100]}')
print('Step 4 完成')

## Step 5: 数据预处理

In [ ]:
import subprocess, json

# 关键：工作目录必须是 GPT_SoVITS/ 才能正确 import
SCRIPT_DIR = os.path.join(REPO_DIR, 'GPT_SoVITS')

pretrained_dir = os.path.abspath('pretrained_models')
bert_dir = f'{pretrained_dir}/chinese-roberta-wwm-ext-large'
hubert_dir = f'{pretrained_dir}/chinese-hubert-base'
pretrained_s2G = f'{pretrained_dir}/gsv-v2final-pretrained/s2G2333k.pth'

# 生成 3-get-semantic 需要的 s2 配置
s2_cfg = {
    'data': {
        'filter_length': 2048, 'hop_length': 640,
        'sampling_rate': 32000, 'n_speakers': 300,
        'cleaned_text': True
    },
    'train': {'segment_size': 20480},
    'model': {'version': 'v2'},
}
S2_CFG_PATH = f'{WORK_DIR}/s2_preprocess.json'
with open(S2_CFG_PATH, 'w') as f:
    json.dump(s2_cfg, f)

print(f'bert_dir: {bert_dir}')
print(f'hubert_dir: {hubert_dir}')
print(f'pretrained_s2G: {pretrained_s2G}')
print(f'inp_text: {INP_TEXT}')
print(f'inp_wav_dir: {os.path.abspath(f"{EXP_DIR}/5-wav32k")}')

base_env = {
    **os.environ,
    'inp_text': INP_TEXT,
    'inp_wav_dir': os.path.abspath(f'{EXP_DIR}/5-wav32k'),
    'exp_name': EXP_NAME,
    'opt_dir': EXP_DIR,
    'i_part': '0',
    'all_parts': '1',
    'is_half': 'True',
}

# 5a: 提取文本特征
print('\n=== 5a: 提取文本特征 ===')
env1 = {**base_env, 'bert_pretrained_dir': bert_dir}
r = subprocess.run(
    [sys.executable, '-s', 'prepare_datasets/1-get-text.py'],
    env=env1, capture_output=True, text=True, cwd=SCRIPT_DIR
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1500:])
else:
    print('5a 完成')

# 5b: 提取 HuBERT 特征
print('\n=== 5b: 提取音频特征 ===')
env2 = {**base_env, 'cnhubert_base_dir': hubert_dir}
r = subprocess.run(
    [sys.executable, '-s', 'prepare_datasets/2-get-hubert-wav32k.py'],
    env=env2, capture_output=True, text=True, cwd=SCRIPT_DIR
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1500:])
else:
    print('5b 完成')

# 5c: 提取语义
print('\n=== 5c: 提取语义 ===')
env3 = {
    **base_env,
    'cnhubert_base_dir': hubert_dir,
    'pretrained_s2G': pretrained_s2G,
    's2config_path': os.path.abspath(S2_CFG_PATH),
}
r = subprocess.run(
    [sys.executable, '-s', 'prepare_datasets/3-get-semantic.py'],
    env=env3, capture_output=True, text=True, cwd=SCRIPT_DIR
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1500:])
else:
    print('5c 完成')

# 复制语义文件
src = f'{EXP_DIR}/6-name2semantic-0.tsv'
dst = f'{EXP_DIR}/6-name2semantic.tsv'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'语义文件: {src} -> {dst}')
else:
    print(f'WARNING: 语义文件不存在 {src}')

# 生成 2-name2text.txt (GPT 训练需要)
s1_dir = f'{EXP_DIR}/logs_s1'
os.makedirs(s1_dir, exist_ok=True)
n2t = f'{s1_dir}/2-name2text.txt'
with open(INP_TEXT, 'r', encoding='utf-8') as fin, \
     open(n2t, 'w', encoding='utf-8') as fout:
    for line in fin:
        parts = line.strip().split('|')
        if len(parts) >= 4:
            wav_name = os.path.basename(parts[0]).replace('.wav', '')
            fout.write(f'{wav_name}\t{parts[3]}\n')
print(f'生成 {n2t}')

# 也复制语义到 logs_s1 (GPT 训练读这个路径)
shutil.copy2(dst, f'{s1_dir}/6-name2semantic.tsv')
print(f'复制语义到 {s1_dir}/6-name2semantic.tsv')

# 检查
print('\n=== 产出 ===')
for sub in ['2a-asr', '3-bert', '4-cnhubert', '5-wav32k', '6-name2semantic']:
    p = f'{EXP_DIR}/{sub}'
    if os.path.exists(p):
        n = len(os.listdir(p))
        print(f'  {sub}: {n} 文件')

print('\nStep 5 完成')

## Step 6: 训练 SoVITS

In [ ]:
os.chdir(REPO_DIR)

# 加载官方配置模板并修改
with open('GPT_SoVITS/configs/s2.json') as f:
    s2 = json.load(f)

s2['train']['batch_size'] = 4
s2['train']['epochs'] = 200
s2['train']['save_every_epoch'] = 50
s2['train']['if_save_latest'] = True
s2['train']['if_save_every_weights'] = True
s2['train']['fp16_run'] = True
s2['train']['grad_ckpt'] = True
s2['train']['gpu_numbers'] = '0'
s2['train']['pretrained_s2G'] = os.path.abspath('pretrained_models/gsv-v2final-pretrained/s2G2333k.pth')
s2['train']['pretrained_s2D'] = os.path.abspath('pretrained_models/gsv-v2final-pretrained/s2D2333k.pth')
s2['data']['exp_dir'] = EXP_DIR
s2['s2_ckpt_dir'] = EXP_DIR
s2['name'] = EXP_NAME
s2['model']['version'] = 'v2'

S2_CFG = f'{WORK_DIR}/s2_config.json'
with open(S2_CFG, 'w') as f:
    json.dump(s2, f, indent=2)

print(f'SoVITS 训练: batch_size=4, epochs=200, save_every_epoch=50')
!python -s GPT_SoVITS/s2_train.py --config {S2_CFG}

print('Step 6 完成')

## Step 7: 训练 GPT

In [ ]:
import yaml

os.chdir(REPO_DIR)

s1_dir = f'{EXP_DIR}/logs_s1'

s1 = {
    'train_semantic_path': f'{s1_dir}/6-name2semantic.tsv',
    'train_phoneme_path': f'{s1_dir}/2-name2text.txt',
    'output_dir': s1_dir,
    'train': {
        'seed': 1234,
        'epochs': 100,
        'batch_size': 8,
        'save_every_n_epoch': 1,
        'if_save_latest': True,
        'if_save_every_weights': True,
        'half_weights_save_dir': f'{EXP_DIR}/weights',
        'exp_name': EXP_NAME,
        'precision': '16-mixed',
        'gradient_clip': 1.0,
    },
    'optimizer': {
        'lr': 0.01,
        'lr_init': 0.00001,
        'lr_end': 0.0001,
        'warmup_steps': 2000,
        'decay_steps': 40000,
    },
    'data': {
        'max_eval_sample': 8,
        'max_sec': 54,
        'num_workers': 4,
        'pad_val': 1024,
    },
    'model': {
        'vocab_size': 1025,
        'phoneme_vocab_size': 512,
        'embedding_dim': 512,
        'hidden_dim': 512,
        'head': 16,
        'linear_units': 2048,
        'n_layer': 24,
        'dropout': 0,
        'EOS': 1024,
        'random_bert': 0,
    },
    'inference': {
        'top_k': 5,
    },
}

S1_CFG = f'{WORK_DIR}/s1_config.yaml'
with open(S1_CFG, 'w') as f:
    yaml.dump(s1, f, default_flow_style=False)

print(f'GPT 训练: batch_size=8, epochs=100')
!python -s GPT_SoVITS/s1_train.py --config_file {S1_CFG}

print('Step 7 完成')

## Step 8: 打包下载

In [ ]:
os.chdir(REPO_DIR)

print('=== 训练产出 ===')
for root, dirs, files in os.walk('output'):
    for f in sorted(files):
        if f.endswith(('.pth', '.ckpt')):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  {fpath} ({size_mb:.1f} MB)')

MODEL_FILE = f'{WORK_DIR}/daniya_gptsovits_model.tar.gz'
!tar -czf {MODEL_FILE} output/
!ls -lh {MODEL_FILE}

from IPython.display import FileLink, display
display(FileLink(MODEL_FILE))
print('\n完成! 点击链接下载模型')